## Chapter 6: LIFT 

This is the Sage notebook for [**Chapter 6 of enumeration.ca**](http://enumeration.ca/toolbox/lift/) by Stephen Melczer. We examine how to use  Sage to compute terms in series expansions involving LIFT, including how to use rational parameterizations of genus zero algebraic curves to get explicit closed form expressions.

### Using LIFT to compute series expansions

In [1]:
# Introduce the lazy power series ring
S.<x> = LazyPowerSeriesRing(QQ)

In [2]:
# Sage can compute the "reverse" f(x) of a power series f(x), which satisfies g(f(x))=x
# If f/G(f) = x as in LIFT then f is the reverse of x/G(x) 
# Example: the GF of plane trees satisfies f/G(f) = x with G = (1-x)^(-1)
G = 1/(1-x)
f = (x / G).revert()
f

x + x^2 + 2*x^3 + 5*x^4 + 14*x^5 + 42*x^6 + 132*x^7 + O(x^8)

In [3]:
# Number of plane trees with 100 vertices
f[100]

227508830794229349661819540395688853956041682601541047340

In [4]:
# Equals the 99th Catalan number
1/100 * binomial(198,99)

227508830794229349661819540395688853956041682601541047340

In [5]:
# Lambert W function
G = exp(-x)
f = (x / G).revert()
f

x - x^2 + 3/2*x^3 - 8/3*x^4 + 125/24*x^5 - 54/5*x^6 + 16807/720*x^7 + O(x^8)

In [6]:
f[100]

-105879118406787542383540312584955245256423950195312500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000/9881297415446727147594496649775206852319571477668037853762810667968023095834839075329261976769165978884198811117

In [7]:
(-100)^(100-1)/factorial(100)

-105879118406787542383540312584955245256423950195312500000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000/9881297415446727147594496649775206852319571477668037853762810667968023095834839075329261976769165978884198811117

In [8]:
# Annoyingly, the command is "revert" for Lazy Power Series and "reverse" for regular Power Series
A.<x> = PowerSeriesRing(QQ, default_prec=7)
G = exp(-x)
f = (x / G).reverse()
f

x - x^2 + 3/2*x^3 - 8/3*x^4 + 125/24*x^5 - 54/5*x^6 + 16807/720*x^7 + O(x^8)

### LIFT and Rational Parameterizations
This section is adapted from work of [Mireille Bousquet-Mélou](https://www.labri.fr/perso/bousquet/) on rooted planar maps.

In [9]:
# The generating function M(x) for "rooted planar maps" enumerated by number of edges satisfies P(x,M(x)) = 0 where P is
var('x M t')
P = 16*x - 1 + (1-18*x)*M + 27*x^2*M^2
show(P)

27*M^2*x^2 - M*(18*x - 1) + 16*x - 1

In [10]:
# We can get the first terms in the GF by solving the quadratic and expanding
P.solve(M)[0].rhs().series(x,10)

1 + 2*x + 9*x^2 + 54*x^3 + 378*x^4 + 2916*x^5 + 24057*x^6 + 208494*x^7 + 1876446*x^8 + 17399772*x^9 + Order(x^10)

How can we get a closed form? 

**One approach:** Differentiate $P(x,M(x))$ to get ODE for $M$. We can convert this to a recurrence for the coefficient sequence and try to solve the recurrence with an advanced solver. But this requires the advanced recurrence solver!

**Alternative:** Use the use general form of LIFT. Recall that if $x = A(x)/ϕ(A(x))$ then 
$$[x^n]ψ(A(x)) = \frac{1}{n}[u^{n-1}]ψ'(u)ϕ(u)^n$$

Our goal is to find "simple" rational functions $\phi(x)$ and $\psi(x)$ and a series $A(x)$ such that $M(x) = \psi(A(x))$ and $A(x) = x \phi(A(x))$.

We can (try to) do this using a classical tool in algebraic geometry: *rational parameterization*.

In [11]:
# Construct the "algebraic curve" defined by P in Sage
P_curve = Curve(QQ['x,M'](P))
P_curve

Affine Plane Curve over Rational Field defined by 27*x^2*M^2 - 18*x*M + 16*x + M - 1

In [12]:
# A "genus zero" curve has a "rational parameterization"
P_curve.genus()

0

In [13]:
# Compute rational parameterization
rat_param = [factor(SR(k)) for k in P_curve.rational_parameterization()]
show(rat_param)

[-(t + 3)/t^2, (t + 4)*t/(t + 3)^2]

In [14]:
# Check that P(x(t),M(t)) = 0
P.subs(x==rat_param[0], M==rat_param[1]).simplify_full()

0

In [15]:
# We get a nicer form for LIFT by replacing t -> 1/t
L = [k.subs(t==1/t).factor() for k in rat_param]
show(L)

[-(3*t + 1)*t, (4*t + 1)/(3*t + 1)^2]

In [16]:
# The equation x == -(3*t + 1)*t has two solutions in t
sols = solve(x==L[0],t)
show(sols)

[t == -1/6*sqrt(-12*x + 1) - 1/6, t == 1/6*sqrt(-12*x + 1) - 1/6]

In [17]:
# we want to find the one corresponding to our generating function
show(L[1].subs(t==sols[0].rhs()).series(x,3))
show(L[1].subs(t==sols[1].rhs()).series(x,3))

(-1/27)*x^(-2) + 2/3*x^(-1) + (-1) + (-2)*x + (-9)*x^2 + Order(x^3)

1 + 2*x + 9*x^2 + Order(x^3)

In [18]:
# The series we care about has no constant, thus we can apply LIFT to get coefficients 
sols[1].rhs().series(x,3)

(-1)*x + (-3)*x^2 + Order(x^3)

In [19]:
# We have x = -(3*t + 1)*t so
phi = t/L[0]
show(phi)

-1/(3*t + 1)

In [20]:
# And we have M = (4*t + 1)/(3*t + 1)^2 so
psi = L[1]
show(psi)

(4*t + 1)/(3*t + 1)^2

In [21]:
# Verify initial terms from LIFT match what we know (aside from constant)
var('n')
psi_diff = psi.diff().factor()
print(P.solve(M)[0].rhs().series(x,9))
[(psi_diff*phi^k/k).series(t,10).truncate().coefficient(t,k-1) for k in range(1,9)]

1 + 2*x + 9*x^2 + 54*x^3 + 378*x^4 + 2916*x^5 + 24057*x^6 + 208494*x^7 + 1876446*x^8 + Order(x^9)


[2, 9, 54, 378, 2916, 24057, 208494, 1876446]

In [22]:
# Symbolically we want the coefficient of t^(n-1) in 
show(12*t*(-1)^(n+1)*(3*t+1)^(-n-3)/n + 2*(-1)^(n+1)*(3*t+1)^(-n-3)/n)

12*(-1)^(n + 1)*(3*t + 1)^(-n - 3)*t/n + 2*(-1)^(n + 1)*(3*t + 1)^(-n - 3)/n

In [23]:
# Negative binomial theorem implies this equals
show(2*3^(n-1)*binomial(2*n+1,n+2)/n - 12*3^(n-2)*binomial(2*n,n+2)/n)

-12*3^(n - 2)*binomial(2*n, n + 2)/n + 2*3^(n - 1)*binomial(2*n + 1, n + 2)/n

In [24]:
# Pull out common factors
show(2*3^(n-1)/n * (binomial(2*n+1,n+2) - 2*binomial(2*n,n+2)))

-2*3^(n - 1)*(2*binomial(2*n, n + 2) - binomial(2*n + 1, n + 2))/n

In [25]:
# Simplify binomial coefficients gives
m_n = 2*3^n*binomial(2*n,n)/((n+1)*(n+2))
show(m_n)

2*3^n*binomial(2*n, n)/((n + 2)*(n + 1))

In [26]:
[m_n.subs(n==k) for k in range(1,9)]

[2, 9, 54, 378, 2916, 24057, 208494, 1876446]

### A More Complicated Example

In [27]:
# The generating function C(x) for "rooted planar cubic maps" enumerated by number of vertices/2 
# satisfies P2(C(x),x) = 0 where P2 is
var('x C')
P2 = 64*C^3 + (-96*x + 1)*C^2 + (30*x - 1)*C*x - x^3*(27*x - 1)
show(P2)

-(27*x - 1)*x^3 + 64*C^3 - C^2*(96*x - 1) + C*(30*x - 1)*x

In [28]:
P2.solve(C)[2].rhs().series(x,10).simplify_full()

11824384*x^9 + 786432*x^8 + 54912*x^7 + 4096*x^6 + 336*x^5 + 32*x^4 + 4*x^3 + x^2 + (I + 1)*Order(x^10)

In [29]:
# Now the parameterization is very complicated!
P2_curve = Curve(QQ['x,C'](P2))
rat_param = list(P2_curve.rational_parameterization())
show(rat_param)

[(-42055879864320*t^3 - 1807639552*t^2 - 17296*t)/(49836032*t^3 + 406272*t^2 + 1104*t + 1),
 (1089173412548771840*t^4 + 41613186392064*t^3 + 299151616*t^2)/(18339659776*t^4 + 199344128*t^3 + 812544*t^2 + 1472*t + 1)]

In [30]:
# But this is just one parameterization. We can make more rational substitutions to try and simplify.
# Here we make one with parameters a,b,c,d
var('a b c d')
L = [SR(k).subs(t==(a+b*t)/(c+d*t)).factor() for k in rat_param]
show(L)

[-17296*(69552*b*t + d*t + 69552*a + c)*(34960*b*t + d*t + 34960*a + c)*(b*t + a)/(368*b*t + d*t + 368*a + c)^3,
 299151616*(104144*b*t + d*t + 104144*a + c)*(34960*b*t + d*t + 34960*a + c)*(b*t + a)^2/(368*b*t + d*t + 368*a + c)^4]

In [31]:
# We want phi to be simple, so lets cancel the factor of t in the denominator of L[0]
L = [k.subs(d==-368*b).factor() for k in L]
show(L)

[-17296*(69184*b*t + 69552*a + c)*(34592*b*t + 34960*a + c)*(b*t + a)/(368*a + c)^3,
 299151616*(103776*b*t + 104144*a + c)*(34592*b*t + 34960*a + c)*(b*t + a)^2/(368*a + c)^4]

In [32]:
# Now lets set a = 0 so that the numerator has a factor of t
L = [k.subs(a==0) for k in L]
show(L)

[-17296*(69184*b*t + c)*(34592*b*t + c)*b*t/c^3,
 299151616*(103776*b*t + c)*(34592*b*t + c)*b^2*t^2/c^4]

In [33]:
# Now set c = 1 to get integer coefficients
L = [k.subs(c==1) for k in L]
show(L)

[-17296*(69184*b*t + 1)*(34592*b*t + 1)*b*t,
 299151616*(103776*b*t + 1)*(34592*b*t + 1)*b^2*t^2]

In [34]:
# Define b to be a constant that cancels lots of factors
L = [k.subs(b==1/lcm([69184,34592,103776,103776])) for k in L]
show(L)

[-1/216*(t + 6)*(t + 3)*t, 1/1728*(t + 6)*(t + 2)*t^2]

In [35]:
# Cancel a few more factors -- much simpler than what we started with!
L = [k.subs(t==12*t) for k in L]
show(L)

[-(4*t + 1)*(2*t + 1)*t, (6*t + 1)*(2*t + 1)*t^2]

In [36]:
# Now the equation x == t(x) has three solutions, and we need to find the correct one
# NOTE: We don't actually need to solve the equation to compute these series expansion (but that's more advanced)
sols = solve(x==L[0],t)
show(L[1].subs(t==sols[0].rhs()).series(x,5).simplify_full())
show(L[1].subs(t==sols[1].rhs()).series(x,5).simplify_full())
show(L[1].subs(t==sols[2].rhs()).series(x,5).simplify_full())

-64*x^4 - 2*x^2 + 1/2*x + (I + 1)*Order(x^5) - 1/64

32*x^4 - 4*x^3 + x^2 + x + (I + 1)*Order(x^5)

32*x^4 + 4*x^3 + x^2 + (I + 1)*Order(x^5)

In [37]:
# The series we care about has no constant, thus we need to make a final change of variables to apply LIFT
sols[2].rhs().series(x,3).simplify_full()

-6*x^2 - x + (I + 1)*Order(x^3)

In [38]:
# Now we have the following
phi = t/L[0]
show(phi)

-1/((4*t + 1)*(2*t + 1))

In [39]:
psi = L[1]
show(psi)

(6*t + 1)*(2*t + 1)*t^2

In [40]:
# Verify initial terms from LIFT match what we know (aside from constant)
var('n')
psi_diff = psi.diff().factor()
print(P2.solve(C)[2].rhs().series(x,10).simplify_full())
[(psi_diff*phi^k/k).series(t,10).truncate().coefficient(t,k-1) for k in range(1,10)]

11824384*x^9 + 786432*x^8 + 54912*x^7 + 4096*x^6 + 336*x^5 + 32*x^4 + 4*x^3 + x^2 + (I + 1)*Order(x^10)


[0, 1, 4, 32, 336, 4096, 54912, 786432, 11824384]

In [41]:
# Thus we want the coefficient of t^(n-1) in the following
show(48*t^3/n/(-(4*t + 1)*(2*t + 1))^n + 24*t^2/n/(-(4*t + 1)*(2*t + 1))^n + 2*t/n/(-(4*t + 1)*(2*t + 1))^n)

48*t^3/((-(4*t + 1)*(2*t + 1))^n*n) + 24*t^2/((-(4*t + 1)*(2*t + 1))^n*n) + 2*t/((-(4*t + 1)*(2*t + 1))^n*n)

In [42]:
# Extracting each of the terms in this sum gives a sum of products of binomial coefficients
# Simplifying that large sum can be done with algorithms for sums of binomials 
# (see the book A=B by Petkovsek, Wilf, and Zeilberger). Ultimately one can obtain the following (up to a shift).
show(2^(3*n + 1)*binomial(n*3/2, n)/((n + 1)*(n + 2)))

2^(3*n + 1)*binomial(3/2*n, n)/((n + 2)*(n + 1))

In [43]:
[2^(3*n + 1)*binomial(n*3/2, n)/((n + 1)*(n + 2)) for n in range(0,10)]

[1, 4, 32, 336, 4096, 54912, 786432, 11824384, 184549376, 2966845440]